In [89]:
# Grouping the multi-indexed df
import pandas as pd
s = pd.Series({'ckm' : 'coffee', 'blr' : 'tech', 'mys' : 'kings'})
df = s.to_frame('famous')
df['temp'] = ['cold', 'hot', 'avg']
df

,famous,temp
ckm,coffee,cold
blr,tech,hot
mys,kings,avg


In [90]:
# Add a new tuple using .loc[]
df.loc['has'] = ['malls', 'avg']
df.loc['dvg'] = ['peda', 'hot']
df

,famous,temp
ckm,coffee,cold
blr,tech,hot
mys,kings,avg
has,malls,avg
dvg,peda,hot


In [91]:
df = df.reset_index()
df = df.sort_values(by='temp')
df = df.set_index(['temp', 'index'])
df

famous
temp index        
avg  mys     kings
     has     malls
cold ckm    coffee
hot  blr      tech
     dvg      peda

In [92]:
for group, frame in df.groupby(level = [0,1]):
    print(group)

('avg', 'has')
('avg', 'mys')
('cold', 'ckm')
('hot', 'blr')
('hot', 'dvg')


In [93]:
for group, frame in df.groupby(level = (0,1)):
    print(group)
    print('\n', frame)
    print('--------------')

('avg', 'has')

            famous
temp index       
avg  has    malls
--------------
('avg', 'mys')

            famous
temp index       
avg  mys    kings
--------------
('cold', 'ckm')

             famous
temp index        
cold ckm    coffee
--------------
('hot', 'blr')

            famous
temp index       
hot  blr     tech
--------------
('hot', 'dvg')

            famous
temp index       
hot  dvg     peda
--------------


In [94]:
# Groupby using only on temp
for group, frame in df.groupby('temp') :
    print(group)
    print('\n', frame)
    print('--------------')

avg

            famous
temp index       
avg  mys    kings
     has    malls
--------------
cold

             famous
temp index        
cold ckm    coffee
--------------
hot

            famous
temp index       
hot  blr     tech
     dvg     peda
--------------


In [95]:
# Teacher dataframe
import numpy as np

d = {
    'sunitha' : ['hod', 5, 1, True],
    'sathya' : ['asso', 4.5, 2, True],
    'sana ara' : ['assi', 3, 3, True],
    'sonia' : ['assi', 4.5, 4, True],
    'purshi' : ['assi', 3, 3, False],
    'bhavya' : ['assi', 4, 3, False],
    'gowri'  : ['assi', np.nan, 4, True]
}

ndf = pd.DataFrame(d)
ndf = ndf.T
ndf = ndf.reset_index()
ndf.columns = ['name', 'post', 'rating', 'rank', 'like / dislike']
ndf

,name,post,rating,rank,like / dislike
0,sunitha,hod,5,1,True
1,sathya,asso,4.5,2,True
2,sana ara,assi,3,3,True
3,sonia,assi,4.5,4,True
4,purshi,assi,3,3,False
5,bhavya,assi,4,3,False
6,gowri,assi,NaN,4,True


In [96]:
# Sort on the basis of rating
ndf = ndf.dropna()
ndf = ndf.sort_values(by='rating', ascending=False)

for group, frame in ndf.groupby('rating') :
    print(group)

3.0
4.0
4.5
5.0


In [97]:
ndf = ndf.set_index(['rating', 'name'])
ndf

post rank like / dislike
rating name                              
5      sunitha    hod    1           True
4.5    sathya    asso    2           True
       sonia     assi    4           True
4      bhavya    assi    3          False
3      sana ara  assi    3           True
       purshi    assi    3          False

In [98]:
for group, frame in ndf.groupby(['rating', 'name']):
    print(group)

(3.0, 'purshi')
(3.0, 'sana ara')
(4.0, 'bhavya')
(4.5, 'sathya')
(4.5, 'sonia')
(5.0, 'sunitha')


In [99]:
# Advanced Grouping technique using custom function
'''
    5 and above 5 -> Fantastic
    4 and above 4 -> Excellent
    else -> Needs improvement
'''

def comment(item):
    if item[0] >= 5:
        return (item[1],'Fantastic')

    elif item[0] >= 4:
        return (item[1], 'Excellent')

    else:
        return (item[1], 'Needs Improvement')

for group, frame in ndf.groupby(by = comment):
    print(group)

('bhavya', 'Excellent')


('purshi', 'Needs Improvement')
('sana ara', 'Needs Improvement')
('sathya', 'Excellent')
('sonia', 'Excellent')
('sunitha', 'Fantastic')


In [100]:
# 3 Steps of Grouping Data
# Aggregation, Transformation, Filtration

### Aggregation

In [101]:
ndf = ndf.reset_index()
ndf

,rating,name,post,rank,like / dislike
0,5.0,sunitha,hod,1,True
1,4.5,sathya,asso,2,True
2,4.5,sonia,assi,4,True
3,4.0,bhavya,assi,3,False
4,3.0,sana ara,assi,3,True
5,3.0,purshi,assi,3,False


In [102]:
(
    ndf
        .groupby('post')['rating']
        .agg(['mean', 'count'])
        .round(2)
        .set_axis(['Average Rating to each post', 'Number of Employees'], axis='columns')
        .sort_values(by='Average Rating to each post', ascending=False)
)

,Average Rating to each post,Number of Employees
post,,
hod,5.00,1
asso,4.50,1
assi,3.62,4


### Transformation

In [103]:
ndf

,rating,name,post,rank,like / dislike
0,5.0,sunitha,hod,1,True
1,4.5,sathya,asso,2,True
2,4.5,sonia,assi,4,True
3,4.0,bhavya,assi,3,False
4,3.0,sana ara,assi,3,True
5,3.0,purshi,assi,3,False


In [104]:
# Aggregation gives out One Result per Group
# Hence using Aggregation results in the change in the shape

ndf.groupby('post')['rating'].transform('mean')

0    5.000
1    4.500
2    3.625
3    3.625
4    3.625
5    3.625
Name: rating, dtype: float64

In [105]:
# Transform helps making changes directly to the df

ndf['Avg Rating to the post'] = ndf.groupby('post')['rating'].transform('mean')
ndf

,rating,name,post,rank,like / dislike,Avg Rating to the post
0,5.0,sunitha,hod,1,True,5.000
1,4.5,sathya,asso,2,True,4.500
2,4.5,sonia,assi,4,True,3.625
3,4.0,bhavya,assi,3,False,3.625
4,3.0,sana ara,assi,3,True,3.625
5,3.0,purshi,assi,3,False,3.625


In [106]:
# Create difference bw actual rating and avg rating

ndf['rating diff'] = ndf['rating'] - ndf['Avg Rating to the post']
ndf

,rating,name,post,rank,like / dislike,Avg Rating to the post,rating diff
0,5.0,sunitha,hod,1,True,5.000,0.000
1,4.5,sathya,asso,2,True,4.500,0.000
2,4.5,sonia,assi,4,True,3.625,0.875
3,4.0,bhavya,assi,3,False,3.625,0.375
4,3.0,sana ara,assi,3,True,3.625,-0.625
5,3.0,purshi,assi,3,False,3.625,-0.625


In [107]:
# Performance Review According to the Post

def review(diff):
    if diff == 0:
        return 'Exact'

    elif diff > 0:
        return 'High'

    else:
        return 'Low'

ndf['Performance Review'] = ndf['rating diff'].apply(review)
ndf

,rating,name,post,rank,like / dislike,Avg Rating to the post,rating diff,Performance Review
0,5.0,sunitha,hod,1,True,5.000,0.000,Exact
1,4.5,sathya,asso,2,True,4.500,0.000,Exact
2,4.5,sonia,assi,4,True,3.625,0.875,High
3,4.0,bhavya,assi,3,False,3.625,0.375,High
4,3.0,sana ara,assi,3,True,3.625,-0.625,Low
5,3.0,purshi,assi,3,False,3.625,-0.625,Low


### Filtering

In [108]:
# Suppose we only want groups whose Performance Review is not Low

(
    ndf
        .groupby('post')
        .filter(lambda x : (x['Performance Review'] != 'Low').all())
)



,rating,name,post,rank,like / dislike,Avg Rating to the post,rating diff,Performance Review
0,5.0,sunitha,hod,1,True,5.0,0.0,Exact
1,4.5,sathya,asso,2,True,4.5,0.0,Exact


### Apply Function

In [109]:
# For each post find the teacher having the max rating

def max_rate(df):
    return df.sort_values(by='rating', ascending=False).head(1)

ndf.groupby('post').apply(max_rate)

,,rating,name,rank,like / dislike,Avg Rating to the post,rating diff,Performance Review
post,,,,,,,,
assi,2,4.5,sonia,4,True,3.625,0.875,High
asso,1,4.5,sathya,2,True,4.500,0.000,Exact
hod,0,5.0,sunitha,1,True,5.000,0.000,Exact


# Scales

## 1) Ratio Scale :
- Units are Equally Spaced <br>
- All mathematical operations such as +, -, *, / are valid <br>
- Contains abscence of a value like 0 <br>
- Ex : Height, Weight <br>

## 2) Interval Scale :
- Units are Equally Spaced <br>
- No abscence of value, No absolute 0 <br>
- Some mathematical operations are invalid like division <br>
- Ex : Temperature, Direction <br>

## 3) Ordinal Scale :
- Order of Units is important <br>
- Not evenly spaced <br>
- Ex : Grades such as A+, A, B etc <br>

## 4) Nominal Scale :
- Just categories of data, no order <br>
- Most observed <br>
- Ex : Teams of a sports team <br>